### Task 2: CheatCode

In [ ]:
  Terminal 1:
  POLICY_NAME=WaveArm GT=false
  RUN_DIR=~/projects/Project-Automaton/aic_results/${POLICY_NAME}_$(date +%Y%m%d_%H%M%S)
  mkdir -p "$RUN_DIR"

  docker run -it --rm \
    --name aic_eval \
    --network host \
    --gpus all \
    -e DISPLAY=:0 \
    -e WAYLAND_DISPLAY=wayland-0 \
    -e XDG_RUNTIME_DIR=/mnt/wslg/runtime-dir \
    -e PULSE_SERVER=/mnt/wslg/PulseServer \
    -e GALLIUM_DRIVER=d3d12 \
    -e LD_LIBRARY_PATH=/usr/lib/wsl/lib \
    -e AIC_RESULTS_DIR=/aic_results \
    -v /tmp/.X11-unix:/tmp/.X11-unix \
    -v /mnt/wslg:/mnt/wslg \
    -v /usr/lib/wsl:/usr/lib/wsl \
    -v "$RUN_DIR":/aic_results \
    --device /dev/dxg \
    ghcr.io/intrinsic-dev/aic/aic_eval:latest \
    ground_truth:=$GT start_aic_engine:=true

  Terminal 2 (once you see Retrying...):
  cd ~/projects/Project-Automaton/References/aic
  pixi run ros2 run aic_model aic_model --ros-args -p use_sim_time:=true -p \
  policy:=aic_example_policies.ros.WaveArm

  ---
  CheatCode

  Terminal 1:
  POLICY_NAME=CheatCode GT=true
  RUN_DIR=~/projects/Project-Automaton/aic_results/${POLICY_NAME}_$(date +%Y%m%d_%H%M%S)
  mkdir -p "$RUN_DIR"

  docker run -it --rm \
    --name aic_eval \
    --network host \
    --gpus all \
    -e DISPLAY=:0 \
    -e WAYLAND_DISPLAY=wayland-0 \
    -e XDG_RUNTIME_DIR=/mnt/wslg/runtime-dir \
    -e PULSE_SERVER=/mnt/wslg/PulseServer \
    -e GALLIUM_DRIVER=d3d12 \
    -e LD_LIBRARY_PATH=/usr/lib/wsl/lib \
    -e AIC_RESULTS_DIR=/aic_results \
    -v /tmp/.X11-unix:/tmp/.X11-unix \
    -v /mnt/wslg:/mnt/wslg \
    -v /usr/lib/wsl:/usr/lib/wsl \
    -v "$RUN_DIR":/aic_results \
    --device /dev/dxg \
    ghcr.io/intrinsic-dev/aic/aic_eval:latest \
    ground_truth:=$GT start_aic_engine:=true

  Terminal 2 (once you see Retrying...):
  cd ~/projects/Project-Automaton/References/aic
  pixi run ros2 run aic_model aic_model --ros-args -p use_sim_time:=true -p \
  policy:=aic_example_policies.ros.CheatCode

### Task 3: Gentle Giant

---

  What the AIC actually supports

  The toolkit officially provides mirror environments for all three:

  ┌───────────┬─────────────────────────┬────────────────────────────────────────┬───────────────┐
  │ Simulator │       Provided by       │                Purpose                 │ Parallel RL?  │
  ├───────────┼─────────────────────────┼────────────────────────────────────────┼───────────────┤
  │ Gazebo    │ Intrinsic (official)    │ Evaluation + training                  │ No (single    │
  │           │                         │                                        │ env)          │
  ├───────────┼─────────────────────────┼────────────────────────────────────────┼───────────────┤
  │ Isaac Lab │ NVIDIA collaboration    │ Teleop, imitation learning, RL with    │ Yes           │
  │           │                         │ rsl-rl                                 │ (thousands)   │
  ├───────────┼─────────────────────────┼────────────────────────────────────────┼───────────────┤
  │ MuJoCo    │ Google DeepMind         │ Same ROS 2 interface, data collection, │ Yes (via MJX) │
  │           │ collaboration           │  policy training                       │               │
  └───────────┴─────────────────────────┴────────────────────────────────────────┴───────────────┘

  And they explicitly encourage multi-simulator training for domain randomization —
  "sim-to-sim-to-real transfer."

  What this means for our plan

  My earlier advice was too conservative. The corrected approach:

  - Gazebo = evaluation (always test here, this is what scores you)
  - Isaac Lab = RL training + synthetic data at scale (your 5090 shines here)
  - MuJoCo = fast controller sweeps + alternative physics for domain randomization
  - All three share ROS 2 interfaces — the MuJoCo integration uses the same ros2_control topics, and
  Isaac Lab has teleop + recording built in

  But the sequencing still matters. You need a working perception + controller pipeline first before
  scaling up RL training across simulators. Running 4096 parallel envs of a policy that doesn't work
  yet just gives you 4096 failures faster.

  Revised recommendation

  1. Now: Finish running example policies in Gazebo (you're doing this)
  2. Next: Build your custom policy + dataset logger in Gazebo
  3. Then: Set up Isaac Lab mirror environment (the guide + assets are ready)
  4. Then: Train perception in Isaac Lab (synthetic data), train RL policies at scale, validate in
  Gazebo

  Want to continue with the example policies for now, or would you rather set up the Isaac Lab
  environment in parallel?

---

   ... Can all three do RL? Yes, but at very different scales:


   Simulator // RL capability // Parallel envs on 5090 // AIC compatible?
   Gazebo // Possible but painful - single env, ~1x realtime // 1 // Yes (official eval)
   MuJoCo/MJX // Excellent - MJX runs thousands of envs via JAX on GPU // 4096+ // ...
   Isaac Sim // Excellent - IsaacGym/Lab runs thousands of envs on GPU // 4096+ // ...




---

WHAT THE `AIC` ACTUALLY SUPPORTS
   The toolkit OFFICIALLY PROVIDES MIRRROR ENVIRONMENTS for all three:

   Simulator // Provided by // Purpose // Parallel RL?
   Gazebo // Intrinsic (official) // Evaluation + training // No (single env)
   Isaac Lab // NVIDIA collaboration // Teleop, imitation learning, RL with rsl-rl // Yes...
   MuJoCo // Google Deepmind collaboration // Same ROS 2 interface, data collection, policy training // Yes...

   ---
   And they EXPLICITLY ENCOURAGE multi-simulator training for domain 
   randomisation -- "sim-to-sim-to-real transfer."


WHAT THIS MEANS FOR OUR PLAN
   My earlier advice...

   * GAZEBO = evaluation (always test here, this is what scores you)
   * Isaac Lab = RL training + synthetic data at scale (5090...)
   * MuJoCo = fast controller sweeps + alternative physics for domain randomisation
   * ALL THREE SHARE ROS 2 INTERFACES -- the MuJoCo integration uses the same
     `ros2_control` topics, and Isaac Lab has teleop + recording built in.

   BUT THE SEQUENCING STILL MATTERS. You need a working perception + controller
   pipeline first before scaling up RL training across simulators. Running 4096
   parallel envs of a policy that doesn't work yet just gives you 4096 failures
   faster.   













-- \* MuJoCo = fast controller sweeps + alternative physics for domain randomisation\
   
   "Controller" here means the ROBOT MOTION CONTROLLER -- the software that
   decides how the robot arm moves, not a microcontroller/hardware chip.

   Specifically, in the AIC context, the controller has tunable parameters like:
      * STIFFNESS (how rigidly the arm holds its target position -- e.g., 50 vs 500)
      * DAMPING (how much it resists motion -- e.g., 5 vs 40)
      * APPROACH SPEED (how fast the arm descends toward the port)
      * INSERTION FORCE limits
      * COMPLIANCE GAINS (how much the arm "gives" when it contacts something)

   A "CONTROLLER SWEEP" means systematically testing many combinations of these
   parameters to find what works best. For example: "what stiffness/damping 
   combo lets the cable insert without jamming?"

   WHY MUJOCO IS GOOD FOR THIS: Gazebo runs at ~1x realtime (one trail = 12 
   seconds real time). MuJoCo with MJX can run thousands of simulations in
   parallel on your GPU. So instead of testing 100 parameter combos one-by-one
   in Gazebo (20+ minutes), you test them all simultaneously in MuJoCo in 
   seconds. Then you take the winning parameters back to Gazebo for validation.

   Think of it like: Gentle.... instead of hand-tuning two extremes, you sweep
   the entire space automatically. 

-- RSL-RL is an open-source Reinforcement Learning (RL) library specifically
   tailored for robotics research. Developed by ... it is designed to be 
   lightweight, high-performance framework optimised for GPU-accelerated 
   training of robotic controllers.

   KEY CHARACTERISTICS AND FEATURES
      * MINIMALIST DESIGN: Unlike general-purpose RL frameworks, RLS-RL 
        prioritises a compact, readable, and easily modifiable codebase, 
        allowing researchers to quickly adapt and extend algorithms.
      * ROBOTICS-FIRST ALGORITHMS: The library centers on Proximal Policy 
        Optimisation (PPO), which is the standard for robot learning, along with
        behavior cloning (BC) techniques similar to DAgger.
      * HIGH-THROUGHPUT TRAINING: Optimised for GPU-only training, it allows for
        large-scale parallelised training, enabling compledx policies to be 
        learned in minutes.
      * SIM-TO-REAL FOCUSED: It includes auxiliary techniques like symmetry 
        augmentation (exploiting the robot's physical structure) and 
        curiosity-driven exploration (using Random Network Distillation - RND) 
        to improve performance in sparse-reward environments.
      * INTEGRATION: It is natively integrated with GPU-accelerated simulation
        environments like NVIDIA Isaac Lab, Legged Gym, and MuJoCo Playground.   
